# 01 – VOiCES dataset profiling
_Auto-generated 2025-08-01_

In [5]:
# TODO: inspect clips
from pathlib import Path
try:
    from soundfile import info        # inside 01_profiling.ipynb
except ModuleNotFoundError:
    raise ModuleNotFoundError(
        "python-soundfile is missing – run `poetry add soundfile` and execute again."
    )




In [11]:
import pathlib, itertools
PROJECT_ROOT = pathlib.Path.cwd().parent            # one level up from notebooks/
RAW_ROOT      = PROJECT_ROOT / "data" / "raw" / "voices" / "VOiCES_devkit"
MANIFEST      = PROJECT_ROOT / "data" / "voices_manifest.parquet"
print("Found a wav?", next(RAW_ROOT.rglob("*.wav"), None))

Found a wav? c:\Users\rogtr\Documents\noisy_speech_pipeline\data\raw\voices\VOiCES_devkit\source-16k\train\sp0032\Lab41-SRI-VOiCES-src-sp0032-ch004137-sg0007.wav


Load manifest

In [12]:
import polars as pl

if MANIFEST.exists():
    df = pl.read_parquet(MANIFEST)
else:
    # build manifest from scratch (one row per WAV)
    rows = []
    for wav in RAW_ROOT.rglob("*.wav"):
        rows.append({"path": wav.as_posix()})
    df = pl.DataFrame(rows).with_columns(
        pl.col("path").str.split("/").list.get(-2).alias("mic_dir")
    )
    df.write_parquet(MANIFEST)

print(f"{len(df):,} clips  •  first 5 rows:")
df.head()

20,248 clips  •  first 5 rows:


path,mic_dir
str,str
"""c:/Users/rogtr/Documents/noisy…","""sp0032"""
"""c:/Users/rogtr/Documents/noisy…","""sp0032"""
"""c:/Users/rogtr/Documents/noisy…","""sp0083"""
"""c:/Users/rogtr/Documents/noisy…","""sp0083"""
"""c:/Users/rogtr/Documents/noisy…","""sp0093"""


Quick stats

In [ ]:
import soundfile as sf, tqdm, numpy as np, json, math

def wav_info(path):
    info = sf.info(path)
    dur  = info.frames / info.samplerate
    return dur, info.samplerate, info.channels

DURS, SRs, CHANS = [], [], []
for p in tqdm.tqdm(df["path"]):
    d, sr, ch = wav_info(RAW_ROOT.parent / p)
    DURS.append(d); SRs.append(sr); CHANS.append(ch)

df = df.with_columns(
    pl.Series("duration_sec", DURS),
    pl.Series("sample_rate",  SRs).cast(pl.Int32),
    pl.Series("channels",     CHANS).cast(pl.Int8)
)

print(f"Total hours      : {df['duration_sec'].sum() / 3600:.1f}")
print(f"Median clip (s)  : {df['duration_sec'].median():.2f}")
print(f"Sample-rates     : {df['sample_rate'].unique().to_list()}")
print(f"Channels         : {df['channels'].unique().to_list()}")


100%|██████████| 20248/20248 [11:06<00:00, 30.39it/s]  


Total hours      : 321.9
Median clip (s)  : 15.97
Sample-rates     : [16000]
Channels         : [1]


FileNotFoundError: [Errno 2] No such file or directory: 'notebooks/dur_hist.json'

Audio quality

In [ ]:
"""
Simple proxy: frame-level RMS and clipped-sample ratio.
Good enough to flag obviously broken files.
"""
BAD = []
for p in tqdm.tqdm(df["path"]):
    wav = RAW_ROOT.parent / p
    y, sr = sf.read(wav, dtype="float32")        # y.shape = (n, ch)
    rms = (y ** 2).mean() ** 0.5
    clipped = (abs(y) > 0.99).mean()             # share of samples that clip
    if rms < 0.001 or clipped > 0.02:
        BAD.append({"path": p, "rms": float(rms), "clipped_ratio": float(clipped)})

print(f"Potentially bad clips: {len(BAD)} / {len(df)}")
pl.DataFrame(BAD).head()